In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

## create gold_dim_date

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_dim_date (
  date_key INT,
  ano INT,
  mes STRING,
  month_start DATE
);

In [0]:
%sql
INSERT INTO workspace.default.gold_dim_date
SELECT DISTINCT
  CAST(
    CONCAT(
      CAST(ano AS STRING),
      CASE mes
        WHEN 'JAN' THEN '01'
        WHEN 'FEV' THEN '02'
        WHEN 'MAR' THEN '03'
        WHEN 'ABR' THEN '04'
        WHEN 'MAI' THEN '05'
        WHEN 'JUN' THEN '06'
        WHEN 'JUL' THEN '07'
        WHEN 'AGO' THEN '08'
        WHEN 'SET' THEN '09'
        WHEN 'OUT' THEN '10'
        WHEN 'NOV' THEN '11'
        WHEN 'DEZ' THEN '12'
      END
    ) AS INT
  ) AS date_key,
  ano,
  mes,
  TO_DATE(
    CONCAT(
      CAST(ano AS STRING),
      '-',
      CASE mes
        WHEN 'JAN' THEN '01'
        WHEN 'FEV' THEN '02'
        WHEN 'MAR' THEN '03'
        WHEN 'ABR' THEN '04'
        WHEN 'MAI' THEN '05'
        WHEN 'JUN' THEN '06'
        WHEN 'JUL' THEN '07'
        WHEN 'AGO' THEN '08'
        WHEN 'SET' THEN '09'
        WHEN 'OUT' THEN '10'
        WHEN 'NOV' THEN '11'
        WHEN 'DEZ' THEN '12'
      END,
      '-01'
    )
  ) AS month_start
FROM (
  SELECT ano, mes FROM workspace.default.fuel_dashboard_anp_imports_silver
  UNION
  SELECT ano, mes FROM workspace.default.fuel_dashboard_anp_refinery_silver
) s
WHERE mes IN ('JAN','FEV','MAR','ABR','MAI','JUN','JUL','AGO','SET','OUT','NOV','DEZ');

## gold_dim_product

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_dim_product (
  product_key INT,
  source_product_name STRING,
  canonical_product STRING,
  display_name STRING,
  product_group STRING,
  is_core_fuel BOOLEAN,
  include_in_self_sufficiency_model BOOLEAN,
  yield_factor_default DOUBLE
);

In [0]:
%sql
INSERT INTO workspace.default.gold_dim_product VALUES
(1,  'ÓLEO DIESEL',            'oleo_diesel',            'Óleo Diesel',             'middle_distillates', true,  true,  0.40),
(2,  'GASOLINA A',             'gasolina_a',             'Gasolina A',              'light_products',     true,  true,  0.22),
(3,  'GLP',                    'glp',                    'GLP',                     'light_products',     true,  true,  0.08),
(4,  'QUEROSENE DE AVIAÇÃO',   'querosene_aviacao',      'Querosene de Aviação',    'middle_distillates', true,  true,  0.12),
(5,  'NAFTA',                  'nafta',                  'Nafta',                   'light_products',     true,  true,  0.10),
(6,  'ÓLEO COMBUSTÍVEL',       'oleo_combustivel',       'Óleo Combustível',        'heavy_products',     true,  true,  0.15),
(7,  'ASFALTO',                'asfalto',                'Asfalto',                 'specialties',        false, false, NULL),
(8,  'COQUE',                  'coque',                  'Coque',                   'byproducts',         false, false, NULL),
(9,  'LUBRIFICANTE',           'lubrificante',           'Lubrificante',            'specialties',        false, false, NULL),
(10, 'PARAFINA',               'parafina',               'Parafina',                'specialties',        false, false, NULL),
(11, 'SOLVENTE',               'solvente',               'Solvente',                'specialties',        false, false, NULL),
(12, 'QUEROSENE ILUMINANTE',   'querosene_iluminante',   'Querosene Iluminante',    'specialties',        false, false, NULL),
(13, 'GASOLINA DE AVIAÇÃO',    'gasolina_aviacao',       'Gasolina de Aviação',     'aviation_specialties', false, false, NULL),
(14, 'OUTROS ENERGÉTICOS',     'outros_energeticos',     'Outros Energéticos',      'aggregated_other',   false, false, NULL),
(15, 'OUTROS NÃO ENERGÉTICOS', 'outros_nao_energeticos', 'Outros Não Energéticos',  'aggregated_other',   false, false, NULL);

In [0]:
%sql
SELECT * 
FROM workspace.default.gold_dim_product
ORDER BY product_key;

## gold_dim_refinery

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_dim_refinery (
  refinery_key BIGINT,
  refinaria STRING,
  unidade_da_federacao STRING
);

In [0]:
%sql
INSERT INTO workspace.default.gold_dim_refinery
SELECT
  ROW_NUMBER() OVER (ORDER BY refinaria, unidade_da_federacao) AS refinery_key,
  refinaria,
  unidade_da_federacao
FROM (
  SELECT DISTINCT refinaria, unidade_da_federacao
  FROM workspace.default.fuel_dashboard_anp_refinery_silver
);

In [0]:
%sql
SELECT * 
FROM workspace.default.gold_dim_refinery
ORDER BY refinery_key;

##create the trade balance fact
Your operacao_comercial has only IMPORTAÇÃO and EXPORTAÇÃO, so this becomes simple.

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_fact_trade_balance (
  date_key INT,
  product_key INT,
  imports_volume DOUBLE,
  exports_volume DOUBLE,
  net_imports_volume DOUBLE,
  imports_value DOUBLE,
  exports_value DOUBLE,
  net_trade_value DOUBLE
);

In [0]:
%sql
INSERT INTO workspace.default.gold_fact_trade_balance
SELECT
  CAST(CONCAT(CAST(i.ano AS STRING), 
    LPAD(
      CASE i.mes
        WHEN 'JAN' THEN '1'
        WHEN 'FEV' THEN '2'
        WHEN 'MAR' THEN '3'
        WHEN 'ABR' THEN '4'
        WHEN 'MAI' THEN '5'
        WHEN 'JUN' THEN '6'
        WHEN 'JUL' THEN '7'
        WHEN 'AGO' THEN '8'
        WHEN 'SET' THEN '9'
        WHEN 'OUT' THEN '10'
        WHEN 'NOV' THEN '11'
        WHEN 'DEZ' THEN '12'
        ELSE '0'
      END, 2, '0')
    ) AS INT) AS date_key,
  p.product_key,

  SUM(CASE WHEN i.operacao_comercial = 'IMPORTAÇÃO' THEN i.importado_exportado ELSE 0 END) AS imports_volume,
  SUM(CASE WHEN i.operacao_comercial = 'EXPORTAÇÃO' THEN i.importado_exportado ELSE 0 END) AS exports_volume,

  SUM(CASE WHEN i.operacao_comercial = 'IMPORTAÇÃO' THEN i.importado_exportado ELSE 0 END)
  - SUM(CASE WHEN i.operacao_comercial = 'EXPORTAÇÃO' THEN i.importado_exportado ELSE 0 END) AS net_imports_volume,

  SUM(CASE WHEN i.operacao_comercial = 'IMPORTAÇÃO' THEN i.dispendio_receita ELSE 0 END) AS imports_value,
  SUM(CASE WHEN i.operacao_comercial = 'EXPORTAÇÃO' THEN i.dispendio_receita ELSE 0 END) AS exports_value,

  SUM(CASE WHEN i.operacao_comercial = 'IMPORTAÇÃO' THEN i.dispendio_receita ELSE 0 END)
  - SUM(CASE WHEN i.operacao_comercial = 'EXPORTAÇÃO' THEN i.dispendio_receita ELSE 0 END) AS net_trade_value

FROM workspace.default.fuel_dashboard_anp_imports_silver i
JOIN workspace.default.gold_dim_product p
  ON i.produto = p.source_product_name
GROUP BY
  CAST(CONCAT(CAST(i.ano AS STRING), 
    LPAD(
      CASE i.mes
        WHEN 'JAN' THEN '1'
        WHEN 'FEV' THEN '2'
        WHEN 'MAR' THEN '3'
        WHEN 'ABR' THEN '4'
        WHEN 'MAI' THEN '5'
        WHEN 'JUN' THEN '6'
        WHEN 'JUL' THEN '7'
        WHEN 'AGO' THEN '8'
        WHEN 'SET' THEN '9'
        WHEN 'OUT' THEN '10'
        WHEN 'NOV' THEN '11'
        WHEN 'DEZ' THEN '12'
        ELSE '0'
      END, 2, '0')
    ) AS INT),
  p.product_key;

In [0]:
%sql
SELECT *
FROM workspace.default.gold_fact_trade_balance
ORDER BY date_key, product_key;

In [0]:
%sql
SELECT
  p.display_name,
  SUM(f.imports_volume) AS imports_volume,
  SUM(f.exports_volume) AS exports_volume,
  SUM(f.net_imports_volume) AS net_imports_volume
FROM workspace.default.gold_fact_trade_balance f
JOIN workspace.default.gold_dim_product p
  ON f.product_key = p.product_key
GROUP BY p.display_name
ORDER BY p.display_name;

##create the refinery production fact
4.1 create the table

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_fact_refinery_production (
  date_key INT,
  refinery_key BIGINT,
  product_key INT,
  production_volume DOUBLE
);

In [0]:
%sql
INSERT INTO workspace.default.gold_fact_refinery_production
SELECT
  CAST(CONCAT(CAST(r.ano AS STRING), 
    LPAD(
      CASE r.mes
        WHEN 'JAN' THEN '1'
        WHEN 'FEV' THEN '2'
        WHEN 'MAR' THEN '3'
        WHEN 'ABR' THEN '4'
        WHEN 'MAI' THEN '5'
        WHEN 'JUN' THEN '6'
        WHEN 'JUL' THEN '7'
        WHEN 'AGO' THEN '8'
        WHEN 'SET' THEN '9'
        WHEN 'OUT' THEN '10'
        WHEN 'NOV' THEN '11'
        WHEN 'DEZ' THEN '12'
        ELSE '0'
      END, 2, '0')
    ) AS INT) AS date_key,
  dref.refinery_key,
  dp.product_key,
  SUM(r.producao) AS production_volume
FROM workspace.default.fuel_dashboard_anp_refinery_silver r
JOIN workspace.default.gold_dim_refinery dref
  ON r.refinaria = dref.refinaria
 AND r.unidade_da_federacao = dref.unidade_da_federacao
JOIN workspace.default.gold_dim_product dp
  ON r.produto = dp.source_product_name
GROUP BY
  CAST(CONCAT(CAST(r.ano AS STRING), 
    LPAD(
      CASE r.mes
        WHEN 'JAN' THEN '1'
        WHEN 'FEV' THEN '2'
        WHEN 'MAR' THEN '3'
        WHEN 'ABR' THEN '4'
        WHEN 'MAI' THEN '5'
        WHEN 'JUN' THEN '6'
        WHEN 'JUL' THEN '7'
        WHEN 'AGO' THEN '8'
        WHEN 'SET' THEN '9'
        WHEN 'OUT' THEN '10'
        WHEN 'NOV' THEN '11'
        WHEN 'DEZ' THEN '12'
        ELSE '0'
      END, 2, '0')
    ) AS INT),
  dref.refinery_key,
  dp.product_key;

In [0]:
%sql
SELECT *
FROM workspace.default.gold_fact_refinery_production
ORDER BY date_key, refinery_key, product_key;

In [0]:
%sql
SELECT
  p.display_name,
  SUM(f.production_volume) AS total_production
FROM workspace.default.gold_fact_refinery_production f
JOIN workspace.default.gold_dim_product p
  ON f.product_key = p.product_key
GROUP BY p.display_name
ORDER BY p.display_name;

##create the main supply balance fact


In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.gold_fact_supply_balance (
  date_key INT,
  product_key INT,
  domestic_production_volume DOUBLE,
  imports_volume DOUBLE,
  exports_volume DOUBLE,
  net_imports_volume DOUBLE,
  total_available_supply DOUBLE,
  dependency_ratio DOUBLE,
  yield_factor DOUBLE,
  required_capacity_per_day DOUBLE,
  refinery_equivalent DOUBLE
);

In [0]:
%sql
INSERT INTO workspace.default.gold_fact_supply_balance
WITH prod AS (
  SELECT
    date_key,
    product_key,
    SUM(production_volume) AS domestic_production_volume
  FROM workspace.default.gold_fact_refinery_production
  GROUP BY date_key, product_key
),
trade AS (
  SELECT
    date_key,
    product_key,
    imports_volume,
    exports_volume,
    net_imports_volume
  FROM workspace.default.gold_fact_trade_balance
)
SELECT
  COALESCE(t.date_key, p.date_key) AS date_key,
  COALESCE(t.product_key, p.product_key) AS product_key,

  COALESCE(p.domestic_production_volume, 0) AS domestic_production_volume,
  COALESCE(t.imports_volume, 0) AS imports_volume,
  COALESCE(t.exports_volume, 0) AS exports_volume,
  COALESCE(t.net_imports_volume, 0) AS net_imports_volume,

  COALESCE(p.domestic_production_volume, 0) + COALESCE(t.net_imports_volume, 0) AS total_available_supply,

  CASE
    WHEN (COALESCE(p.domestic_production_volume, 0) + COALESCE(t.net_imports_volume, 0)) > 0
    THEN COALESCE(t.net_imports_volume, 0) / (COALESCE(p.domestic_production_volume, 0) + COALESCE(t.net_imports_volume, 0))
    ELSE 0
  END AS dependency_ratio,

  dp.yield_factor_default AS yield_factor,

  CASE
    WHEN dp.yield_factor_default IS NOT NULL AND COALESCE(t.net_imports_volume, 0) > 0
    THEN (COALESCE(t.net_imports_volume, 0) / 30.0) / dp.yield_factor_default
    ELSE 0
  END AS required_capacity_per_day,

  CASE
    WHEN dp.yield_factor_default IS NOT NULL AND COALESCE(t.net_imports_volume, 0) > 0
    THEN ((COALESCE(t.net_imports_volume, 0) / 30.0) / dp.yield_factor_default) / 14300
    ELSE 0
  END AS refinery_equivalent

FROM trade t
FULL OUTER JOIN prod p
  ON t.date_key = p.date_key
 AND t.product_key = p.product_key
JOIN workspace.default.gold_dim_product dp
  ON COALESCE(t.product_key, p.product_key) = dp.product_key;

In [0]:
%sql
SELECT *
FROM workspace.default.gold_fact_supply_balance
ORDER BY date_key, product_key;

In [0]:
%sql
SELECT
  d.month_start,
  p.display_name,
  s.domestic_production_volume,
  s.imports_volume,
  s.exports_volume,
  s.net_imports_volume,
  s.dependency_ratio,
  s.refinery_equivalent
FROM workspace.default.gold_fact_supply_balance s
JOIN workspace.default.gold_dim_date d
  ON s.date_key = d.date_key
JOIN workspace.default.gold_dim_product p
  ON s.product_key = p.product_key
WHERE p.include_in_self_sufficiency_model = true
ORDER BY d.month_start, p.display_name;

In [0]:
%sql
CREATE OR REPLACE VIEW workspace.default.vw_tableau_core_fuels_balance AS
SELECT
  d.month_start,
  d.ano,
  d.mes,
  p.display_name AS produto,
  p.product_group,
  s.domestic_production_volume,
  s.imports_volume,
  s.exports_volume,
  s.net_imports_volume,
  s.total_available_supply,
  s.dependency_ratio,
  s.yield_factor,
  s.required_capacity_per_day,
  s.refinery_equivalent
FROM workspace.default.gold_fact_supply_balance s
JOIN workspace.default.gold_dim_date d
  ON s.date_key = d.date_key
JOIN workspace.default.gold_dim_product p
  ON s.product_key = p.product_key
WHERE p.include_in_self_sufficiency_model = true;

In [0]:
%sql
SELECT *
FROM workspace.default.vw_tableau_core_fuels_balance
ORDER BY month_start, produto;

In [0]:
%sql
SELECT 
AVG (refinery_equivalent)
FROM workspace.default.vw_tableau_core_fuels_balance 
WHERE MES = 'DEZ' and ANO = 2025 ;

In [0]:
%sql
SELECT 
(refinery_equivalent)
FROM workspace.default.vw_tableau_core_fuels_balance 
WHERE MES = 'DEZ' and ANO = 2025  and produto = 'Óleo Diesel';

In [0]:
%sql
SELECT 
(net_imports_volume)
FROM workspace.default.vw_tableau_core_fuels_balance 
WHERE MES = 'DEZ' and ANO = 2025 AND produto = 'Óleo Diesel' ;

In [0]:
%sql
SELECT mes, ano, yield_factor, produto, net_imports_volume, refinery_equivalent
FROM workspace.default.vw_tableau_core_fuels_balance
WHERE mes = 'DEZ' AND ano = 2025;

In [0]:
%sql

SELECT (*) 
FROM bronze_anp_refinery;

In [0]:
%sql
CREATE or REPLACE TABLE workspace.default.vw_tableau_core_fuels_balance_2000
SELECT *
FROM workspace.default.vw_tableau_core_fuels_balance
WHERE ano >= 2000
ORDER BY month_start;

In [0]:
%sql
SELECT *
FROM workspace.default.vw_tableau_core_fuels_balance_2000;

In [0]:
%sql

SELECT (*)
from workspace.default.vw_tableau_core_fuels_balance_2000
WHERE produto = "Óleo Combustível";